In [0]:
# 必要なライブラリのインストール
%pip install -U -qqqq "databricks-langchain==0.21.0" "langchain>=1.0.0" "langgraph>=1.1.0" "mlflow[databricks]>=3.10.1" "databricks-agents>=1.9.3" "databricks-sdk>=0.124.0"
dbutils.library.restartPython()

# 1-2. Scratch パターン：RAG エージェントの構築とデプロイ

このノートブックでは、1-1 で作った AI Search の索引を検索ツールとして使う **RAG エージェント** を作り、MLflow で動きを確かめて評価し、**Agent Bricks CLI** で **Agent Runtime（Databricks Apps 上）** にデプロイする。

```
ノートブックで試作 ──→ MLflow Tracing で過程を見る ──→ mlflow.genai.evaluate で採点
        │
        └─ agent/（Agent Bricks のプロジェクト）に設定を書き出す
                 └─ Web Terminal で agentbricks deploy ──→ agent-bricks-<名前>（Databricks Apps）
```

## このノートブックでやること
1. パラメータの設定（widget）
2. 索引を検索ツールにする（`VectorSearchRetrieverTool`）
3. LangGraph でエージェントを作り、MLflow Tracing で過程を見る
4. 評価データセットで採点する（`mlflow.genai.evaluate`）
5. Agent Bricks のプロジェクトに設定を書き出し、Web Terminal からデプロイする
6. 質問者本人の権限で検索することを確かめ、チャット UI で試す

## 前提
- **1-1** を実行し、索引 `docs_chunks_index` ができている
- サーバーレスのノートブック（環境バージョン 4 で動作確認）
- Agent Bricks CLI は **Beta**（2026年10月時点）。CLI はノートブックのセルからは動かないため、デプロイだけ **Web Terminal** で実行する

> **参考**：以前は Model Serving や、MLflow `AgentServer` を Databricks Apps に直接デプロイする方法が使われていた。これらは Legacy になり、新しく作るエージェントは Agent Bricks CLI と `DurableAgentServer`（Agent Runtime）が推奨である。

---

## 1-2-1. パラメータの設定

| widget | 内容 |
|---|---|
| `CATALOG_NAME` / `SCHEMA_NAME` / `INDEX_NAME` | 1-1 で作った索引 |
| `LLM_ENDPOINT` | 回答に使う Foundation Model API のエンドポイント |
| `AGENT_NAME` | デプロイ名。アプリ名は `agent-bricks-<AGENT_NAME>` になる。英小文字・数字・ハイフンで17文字まで。参加者ごとに重ならない名前にする |
| `EXPERIMENT_NAME` | トレースと評価を記録する MLflow エクスペリメント。空欄なら自分のホームに作る |

In [0]:
# パラメータ設定（widget から取得）
import os
import re

dbutils.widgets.text("CATALOG_NAME", "main", "カタログ名")
dbutils.widgets.text("SCHEMA_NAME", "rag_handson", "スキーマ名")
dbutils.widgets.text("INDEX_NAME", "docs_chunks_index", "AI Search 索引名")
dbutils.widgets.text("LLM_ENDPOINT", "databricks-claude-sonnet-4-5", "LLM のエンドポイント")
dbutils.widgets.text("AGENT_NAME", "rag-handson", "デプロイ名（17文字まで）")
dbutils.widgets.text("EXPERIMENT_NAME", "", "MLflow エクスペリメント（空欄は自動）")

CATALOG_NAME = dbutils.widgets.get("CATALOG_NAME")
SCHEMA_NAME = dbutils.widgets.get("SCHEMA_NAME")
INDEX_NAME = dbutils.widgets.get("INDEX_NAME")
LLM_ENDPOINT = dbutils.widgets.get("LLM_ENDPOINT")
AGENT_NAME = dbutils.widgets.get("AGENT_NAME").strip()
EXPERIMENT_NAME = dbutils.widgets.get("EXPERIMENT_NAME").strip()

assert re.fullmatch(r"[a-z0-9-]{1,17}", AGENT_NAME), "AGENT_NAME は英小文字・数字・ハイフンで17文字まで"

INDEX_FULLNAME = f"{CATALOG_NAME}.{SCHEMA_NAME}.{INDEX_NAME}"
APP_NAME = f"agent-bricks-{AGENT_NAME}"

ME = spark.sql("SELECT current_user()").first()[0]
if not EXPERIMENT_NAME:
    EXPERIMENT_NAME = f"/Users/{ME}/rag-handson-agent"

# このノートブックと同じフォルダーにある agent/ が Agent Bricks のプロジェクト
NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
AGENT_PROJECT_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH) + "/agent"

print(f"索引: {INDEX_FULLNAME}")
print(f"LLM: {LLM_ENDPOINT}")
print(f"アプリ名: {APP_NAME}")
print(f"エクスペリメント: {EXPERIMENT_NAME}")
print(f"プロジェクト: {AGENT_PROJECT_DIR}")

### MLflow の準備

エクスペリメントを設定し、LangChain／LangGraph の呼び出しを自動でトレースする（`mlflow.langchain.autolog()`）。この後エージェントを動かすと、セルの下にトレースが表示される。

In [0]:
import mlflow

mlflow.set_experiment(EXPERIMENT_NAME)
mlflow.langchain.autolog()
print(f"MLflow エクスペリメント: {EXPERIMENT_NAME}")

## 1-2-2. 索引を検索ツールにする

**AI Bridge** の `VectorSearchRetrieverTool`（`databricks-langchain`）を使うと、AI Search の索引をそのままエージェントのツールにできる。新しく作る Python のエージェントでは、この方法が推奨である。

- `query_type="HYBRID"`：ベクトル検索とキーワード検索を組み合わせる
- `columns`：検索結果として返す列。ファイル名とページ番号を返し、回答で出典を示せるようにする
- `tool_description`：LLM がいつこのツールを使うかを判断する説明

まずツールだけを呼び、返る内容を確かめる。

In [0]:
from databricks_langchain import VectorSearchRetrieverTool

doc_search = VectorSearchRetrieverTool(
    index_name=INDEX_FULLNAME,
    tool_name="search_documents",
    tool_description=(
        "ボリュームに置いた文書（Automotive SPICE PAM 4.0 日本語版など）から、質問に関係する箇所を探す。"
        "結果にはファイル名（file_name）とページ番号（pages）が付く。"
    ),
    num_results=5,
    columns=["chunk_id", "file_name", "pages", "chunk_to_retrieve"],
    query_type="HYBRID",
)

# ツールが LLM に返す内容（検索した文書とメタデータ）をそのまま表示する
result = doc_search.invoke("プロセス能力レベル2のプロセス属性")
print(result[:2000])

## 1-2-3. エージェントを作る

LangGraph の `create_agent` で、LLM と検索ツールを組み合わせたエージェントを作る。LLM は質問を読んで検索ツールを呼び、返った文書を根拠に回答する。システムプロンプトで、根拠のページを示すよう指示する。

ここで作るエージェントは、後でデプロイする `agent/agent/agent.py` と同じ構成（同じ索引・同じ検索方式・同じシステムプロンプト）である。

In [0]:
from databricks_langchain import ChatDatabricks
from langchain.agents import create_agent

SYSTEM_PROMPT = """あなたは社内文書について答えるアシスタントである。
質問に答えるときは、必ず文書検索ツールで関連する箇所を探し、見つかった内容だけを根拠に日本語で答える。
回答の最後に、根拠にしたファイル名とページ番号を「出典: <ファイル名> p.<ページ>」の形で列挙する。
検索しても根拠が見つからない場合は、分からないと答える。"""

llm = ChatDatabricks(endpoint=LLM_ENDPOINT)
agent = create_agent(model=llm, tools=[doc_search], system_prompt=SYSTEM_PROMPT)


def answer_text(message) -> str:
    content = message.content
    if isinstance(content, list):
        return "".join(part.get("text", "") for part in content if isinstance(part, dict))
    return content


def ask(question: str) -> str:
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return answer_text(result["messages"][-1])


print(ask("Automotive SPICE のプロセス能力レベル2を達成するために必要なプロセス属性を教えて。"))

### トレースを確かめる

上のセルの出力の下に **トレース** が表示される。開いて次を確かめる。

- LLM が `search_documents` ツールを呼んでいること
- ツールが返したチャンク（ファイル名・ページ番号付き）
- 最終的な回答と、各ステップの所要時間・トークン数

同じトレースは、左サイドバー「エクスペリメント」→ 1-2-1 で設定したエクスペリメント →「トレース」タブでも見られる。

## 1-2-4. 評価データセットで採点する

`mlflow.genai.evaluate()` は、質問の一覧（評価データセット）でエージェントを動かし、**scorer**（LLM judge やコードによる判定）で採点する。プロンプトや検索の設定を変えたときに、品質が上がったか下がったかを数字で比べられる。

ここでは次の scorer を使う。

| scorer | 見るもの |
|---|---|
| `Correctness` | 期待する事実（`expected_facts`）が回答に含まれるか |
| `RelevanceToQuery` | 回答が質問に答えているか |
| `Guidelines` | 独自のルール（ここでは「日本語で答え、出典のページを示す」）を守っているか |

評価データセットの質問と期待する事実は、サンプル PDF の内容から作った。別の PDF を使う場合は書き換える。

> 回答が検索した文書に基づいているかを見る `RetrievalGroundedness` は、トレースに RETRIEVER 型のスパンが必要である。デプロイ用の `agent/agent/tools/doc_search.py` は RETRIEVER 型のスパンを記録するので、デプロイ後のトレースの評価やモニタリングで使える。

In [0]:
eval_data = [
    {
        "inputs": {"question": "プロセス能力レベル2を構成するプロセス属性は何か。"},
        "expectations": {"expected_facts": ["PA 2.1 実施管理プロセス属性", "PA 2.2 作業成果物管理プロセス属性"]},
    },
    {
        "inputs": {"question": "プロセス能力レベルはいくつあり、それぞれ何と呼ばれるか。"},
        "expectations": {"expected_facts": [
            "能力レベルは0から5までの6つ",
            "レベル0は不完全なプロセス、レベル1は実施されたプロセス、レベル2は管理されたプロセス",
            "レベル3は確立されたプロセス、レベル4は予測可能なプロセス、レベル5は革新しているプロセス",
        ]},
    },
    {
        "inputs": {"question": "プロセス属性の評定 N・P・L・F は、それぞれどの程度の達成を表すか。"},
        "expectations": {"expected_facts": [
            "N は達成していない（0〜15%）",
            "P は部分的に達成している（15%超〜50%）",
            "L はおおむね達成している（50%超〜85%）",
            "F は十分に達成している（85%超〜100%）",
        ]},
    },
    {
        "inputs": {"question": "SWE.1 ソフトウェア要求分析のプロセス目的は何か。"},
        "expectations": {"expected_facts": [
            "システム要求およびシステムアーキテクチャと一貫性のあるソフトウェア要求の集合を確立する",
            "ソフトウェア要求は構造化および分析されている",
        ]},
    },
    {
        "inputs": {"question": "能力レベル2を達成するには、各プロセス属性がどう評定される必要があるか。"},
        "expectations": {"expected_facts": [
            "PA 1.1 は十分に達成（F）",
            "PA 2.1 と PA 2.2 はおおむね達成（L）または十分に達成（F）",
        ]},
    },
]


def predict_fn(question: str) -> str:
    return ask(question)

In [0]:
from mlflow.genai.scorers import Correctness, Guidelines, RelevanceToQuery

scorers = [
    Correctness(),
    RelevanceToQuery(),
    Guidelines(name="japanese_with_source", guidelines="回答は日本語で書かれ、根拠のファイル名とページ番号を示している。"),
]

eval_result = mlflow.genai.evaluate(data=eval_data, predict_fn=predict_fn, scorers=scorers)
display(spark.createDataFrame([(k, float(v)) for k, v in eval_result.metrics.items()], "metric STRING, value DOUBLE"))

評価の結果は、エクスペリメントの「評価」タブで質問ごとに見られる。どの scorer がなぜその判定にしたか（judge の理由）も確かめる。

> **やってみよう**：システムプロンプトや `num_results`、`query_type` を変えて 1-2-3 と 1-2-4 を再実行し、評価の結果を比べる。

## 1-2-5. Agent Bricks のプロジェクトに設定を書き出す

このノートブックと同じフォルダーの `agent/` が、`agentbricks init` で作った LangGraph のプロジェクトである。ハンズオン用に次を変えてある。

| ファイル | 内容 |
|---|---|
| `agent/agent/agent.py` | LLM のエンドポイント（`LLM_ENDPOINT`）とシステムプロンプト。ここで試したものと同じ |
| `agent/agent/doc_search.py` | 同じ索引を同じ列・同じ方式（HYBRID）で検索するツール。リクエストごとに **質問したユーザー本人の資格情報** を受け取り、SDK の `query_index` で検索する |
| `agent/runtime/main.py` | `DurableAgentServer` を起動する（テンプレートのまま） |
| `agent/app.yaml` | アプリの起動コマンドと環境変数（索引名・LLM） |
| `agent/agent.toml` | デプロイ名、トレースの送り先、ユーザー権限の設定（`[auth.user]`、スコープ `vector-search`） |

次のセルで、widget の値を `app.yaml` と `agent.toml` に書き込む。

In [0]:
from pathlib import Path

project = Path(AGENT_PROJECT_DIR)
assert (project / "runtime" / "main.py").exists(), f"Agent Bricks のプロジェクトが見つからない: {project}"

(project / "app.yaml").write_text(f"""command: ["uv", "run", "start-server"]
# 1-2 のノートブックが書き込んだ設定
env:
  - name: AI_SEARCH_INDEX
    value: "{INDEX_FULLNAME}"
  - name: LLM_ENDPOINT
    value: "{LLM_ENDPOINT}"
""")

(project / "agent.toml").write_text(f"""schema_version = 1

[agent]
framework = "langgraph"
server = "agentbricks"
deployment_name = "{AGENT_NAME}"

# ハンズオンでは長期記憶（memory store）と会話履歴（session store）は使わない。

# 検索ツールは、質問したユーザー本人の権限で AI Search の索引を検索する
[auth.user]
required = true
additional_api_scopes = ["vector-search"]

[tracing]
experiment_name = "{EXPERIMENT_NAME}"
""")

print((project / "app.yaml").read_text())
print((project / "agent.toml").read_text())

## 1-2-6. Web Terminal からデプロイする

`agentbricks deploy` は、プロジェクトのソースを Databricks Apps にデプロイし、Runtime Store とトレースの設定を自動で用意する。初回は数分かかる。

1. ノートブック右側のパネルで **ターミナル**（Web Terminal）を開く
2. 次のセルが出力するコマンドを、ターミナルに貼り付けて実行する
3. 最後に表示される URL（`agent-bricks-<名前>`）を控える

- `--allow-user-scope-update` は、アプリにユーザー権限のスコープ（`vector-search`）を追加することを許可するオプションである。初回のデプロイで必要になる
- 同じ `AGENT_NAME` で以前の版をデプロイしたことがある場合、デプロイ先のフォルダー（`/Users/<自分>/agentbricks_deployments/agent-bricks-<名前>`）に古いファイルが残ることがある。動きがおかしいときは、`agentbricks deployments delete agent-bricks-<名前> --delete-source` で消してからデプロイし直す

> **ローカル PC から実行する場合**：`databricks auth login --profile <名前>` で認証し、リポジトリを手元に clone してから、`agent/` で `agentbricks --profile <名前> deploy` を実行する。

In [0]:
print(f"""# --- Web Terminal に貼り付けて実行する ---
pip install -q -U databricks-agentbricks
cd "{AGENT_PROJECT_DIR}"
agentbricks deploy --allow-user-scope-update
agentbricks deployments get {APP_NAME}
""")

## 1-2-7. エージェントは質問者本人の権限で検索する

`agent.toml` の `[auth.user]` により、デプロイしたエージェントの検索ツールは、**質問したユーザー本人の権限** で AI Search の索引を検索する（user authorization）。

- アプリのサービスプリンシパルに索引の権限を付ける必要はない。利用者ごとの Unity Catalog の権限（索引の `SELECT` など）がそのまま効く
- 1-1 で索引を作り直しても、権限を付け直す必要はない
- 初めてチャット UI を開くと、アプリが利用者に代わって AI Search を使うことへの **同意画面** が出る。「許可」をクリックする
- LLM の呼び出しとトレースの記録は、アプリのサービスプリンシパルで行う（デプロイ時に自動で設定される）

> ワークスペース管理者が「設定」→「開発」→「Apps」でユーザー権限のスコープを制限している場合は、`vector-search` を許可してもらう。

デプロイの結果、アプリにユーザー権限のスコープが付いたことを確かめる。

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
app = w.apps.get(APP_NAME)
print(f"アプリ: {APP_NAME}（{app.url}）")
print(f"ユーザー権限のスコープ: {app.effective_user_api_scopes or app.user_api_scopes}")

## 1-2-8. デプロイしたエージェントを試す

アプリの URL をブラウザで開くと、テンプレートに付いている **チャット UI** でエージェントと対話できる。サンプルの質問ボタンを押すか、自分で質問を入力する。

- 回答の最後に出典（ファイル名とページ）が付くこと
- エクスペリメントの「トレース」タブに、デプロイ後の呼び出しも記録されること

を確かめる。外部のアプリからは、利用者の Databricks OAuth トークン（U2M）を付けて `POST <アプリの URL>/api/invocations` を呼ぶ。このエージェントは本人の権限で検索するため、別の Databricks App から呼ぶ場合は利用者のトークンが引き継がれない点に注意する（演習3の切り替えアプリは、同じ索引と同じ LLM を直接呼んでいる）。

In [0]:
print(f"チャット UI: {w.apps.get(APP_NAME).url}")

## まとめ

- `VectorSearchRetrieverTool` で索引を検索ツールにし、LangGraph でエージェントを作った
- MLflow Tracing で検索と回答の過程を見て、`mlflow.genai.evaluate()` で採点した
- Agent Bricks CLI で Agent Runtime（Databricks Apps 上）にデプロイし、チャット UI で試した

### 次のステップ
- **2_Managed_Genie AgentとContent Search** で、同じ PDF をノーコードで扱う Managed パターンを試す
- 構造化データも扱いたい場合は、`agentbricks tools add genie-agent` で Genie Agent をこのエージェントのツールに追加できる（ハイブリッド構成）

### 後片付け
README の「後片付け」に従う。このノートブックで作ったものは、アプリ（`agentbricks deployments delete`）と MLflow エクスペリメントである。

### 参考リンク
- [Connect agents to unstructured data（AI Bridge）](https://docs.databricks.com/aws/ja/agents/custom-agents/unstructured-retrieval-tools)
- [Agent Bricks CLI](https://docs.databricks.com/aws/ja/agents/custom-agents/agent-bricks-cli)
- [Agent Runtime](https://docs.databricks.com/aws/ja/agents/deploy/agent-runtime)
- [MLflow：評価とモニタリング](https://docs.databricks.com/aws/ja/mlflow3/genai/eval-monitor/)